# 1. Exploratory Data Analysis (EDA)
## ICU XAI Research - Understanding Patient Data

This notebook performs exploratory analysis on ICU patient data to understand:
- Data distributions and summary statistics
- Missing value patterns
- Feature correlations
- Patient outcome distributions

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from load_data import load_raw_data

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)


In [ ]:
# Load raw ICU data from the PhysioNet set-a files
data_dir = Path.cwd().parent / 'data'
raw_df = load_raw_data(data_dir, dataset='set-a', outcomes_file='Outcomes-a.txt')

print(f"Raw data shape: {raw_df.shape}")
print(f"Unique patients: {raw_df['RecordID'].nunique()}")
print(raw_df[['RecordID', 'In-hospital_death', 'SAPS-I', 'SOFA']].drop_duplicates().head())


In [ ]:
# Basic statistics and missingness
print(raw_df.describe(include='all'))
print("\nLabel distribution:\n", raw_df[['RecordID', 'In-hospital_death']].drop_duplicates()['In-hospital_death'].value_counts())
print("\nMissing values per parameter:\n", raw_df.groupby('Parameter')['Value'].apply(lambda s: s.isna().sum()).sort_values(ascending=False).head(20))


In [ ]:
# Outcome distribution and SAPS/SOFA histograms
outcomes = raw_df[['RecordID', 'In-hospital_death', 'SAPS-I', 'SOFA']].drop_duplicates()
ax = outcomes['In-hospital_death'].value_counts().sort_index().plot(kind='bar')
ax.set_title('ICU Mortality Outcome Distribution')
ax.set_xlabel('In-hospital death')
ax.set_ylabel('Patient count')
plt.show()

sns.histplot(outcomes['SAPS-I'].dropna(), kde=True)
plt.title('SAPS-I Score Distribution')
plt.xlabel('SAPS-I')
plt.show()

sns.histplot(outcomes['SOFA'].dropna(), kde=True)
plt.title('SOFA Score Distribution')
plt.xlabel('SOFA')
plt.show()


In [ ]:
# Patient trajectory visualization for a sample record
sample_id = raw_df['RecordID'].unique()[0]
sample_df = raw_df[raw_df['RecordID'] == sample_id]
patient_ts = sample_df.pivot(index='TimeMinutes', columns='Parameter', values='Value')
patient_ts[['HR', 'NISysABP', 'NIDiasABP', 'Temp']].plot(subplots=True, layout=(2, 2), figsize=(14, 10), title=f'Patient {sample_id} Vital Signs Over Time')
plt.tight_layout()
plt.show()


In [ ]:
# Correlation among outcome labels and clinical severity scores
corr_df = outcomes[['In-hospital_death', 'SAPS-I', 'SOFA']].dropna()
print(corr_df.corr())
sns.heatmap(corr_df.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Among In-hospital Death, SAPS-I, and SOFA')
plt.show()
